# WCCI Action Distribution and Checkpoint-Style Plots

This notebook mirrors the A0 full-test/checkpoint action-distribution notebook, but targets the downloaded WCCI groups. It uses W&B history action metrics by default and automatically picks up WCCI full-test `action_summary.json` files if they are added later.

In [1]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def find_task_dir(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "main.py").is_file() and candidate.name == "Topology_Task":
            return candidate
        task_dir = candidate / "Topology_Task"
        if (task_dir / "main.py").is_file():
            return task_dir
    raise RuntimeError("Could not locate Topology_Task/main.py")


TASK_DIR = find_task_dir()
HELPER_DIR = TASK_DIR / "analysis" / "metrics" / "helpers"
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

from run_data import scan_run_data

RUN_DATA_ROOT = TASK_DIR / "outputs" / "run_data"
FIG_DIR = TASK_DIR / "outputs" / "wcci_metric_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

WCCI_GROUPS = [
    "wcci_aib_24h",
    "wcci_sparse16",
    "wcci_hvg",
    "wcci_baseline",
    "wcci_aib",
]
SEEDS = (0, 1, 2)
AGENTS = ["agent_0", "agent_1", "agent_2", "agent_3"]
SAVE_FIGURES = True
SHOW_FIGURES = True

print("Task dir:", TASK_DIR)
print("Run data root:", RUN_DATA_ROOT)
print("WCCI groups:", WCCI_GROUPS)


Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Run data root: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data
WCCI groups: ['wcci_aib_24h', 'wcci_sparse16', 'wcci_hvg', 'wcci_baseline', 'wcci_aib']


## Load Downloaded WCCI Histories

In [2]:
def _path_or_none(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    path = Path(value).expanduser()
    return path if path.exists() else None


def read_history(row):
    parquet_path = _path_or_none(row.get("history_parquet"))
    csv_path = _path_or_none(row.get("history_csv"))
    if parquet_path is not None:
        history = pd.read_parquet(parquet_path)
    elif csv_path is not None:
        history = pd.read_csv(csv_path)
    else:
        raise FileNotFoundError(f"No local history file for {row.get('run_name')}")
    history = history.copy()
    history["run_name"] = row["run_name"]
    history["run_id"] = row["run_id"]
    history["download_group"] = row["download_group"]
    history["run_state"] = row.get("state")
    return history


run_index_all = scan_run_data(RUN_DATA_ROOT, include_legacy=False)
run_index = run_index_all[run_index_all["download_group"].isin(WCCI_GROUPS)].copy()
if run_index.empty:
    raise RuntimeError(f"No WCCI runs found under {RUN_DATA_ROOT}. Check that the downloaded groups exist.")

histories = []
for row in run_index.to_dict("records"):
    try:
        histories.append(read_history(row))
    except Exception as exc:
        print(f"Skipped {row.get('run_name')}: {exc}")

history_df = pd.concat(histories, ignore_index=True, sort=False) if histories else pd.DataFrame()
if history_df.empty:
    raise RuntimeError("No WCCI histories could be loaded.")

coverage = (
    run_index.assign(
        family=lambda df: df["run_name"].map(lambda name: re.sub(r"_s\\d+$", "", str(name))),
        seed=lambda df: df["run_name"].map(lambda name: int(re.search(r"_s(\\d+)$", str(name)).group(1)) if re.search(r"_s(\\d+)$", str(name)) else np.nan),
    )
    .groupby(["download_group", "family", "state"], dropna=False, as_index=False)
    .agg(
        runs=("run_name", "count"),
        seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        rows_min=("rows", "min"),
        rows_max=("rows", "max"),
    )
    .sort_values(["download_group", "family", "state"])
)
print(f"Loaded {len(run_index)} runs and {len(history_df):,} history rows.")
display(coverage)


Loaded 51 runs and 54,798 history rows.


,download_group,family,state,runs,seeds,rows_min,rows_max
0,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,finished,1,[],1446,1446
1,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,finished,1,[],1446,1446
2,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,finished,1,[],1446,1446
3,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s0,finished,1,[],1446,1446
4,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s1,finished,1,[],1446,1446
5,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s2,finished,1,[],1446,1446
6,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s0,finished,1,[],1446,1446
7,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s1,finished,1,[],1446,1446
8,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s2,finished,1,[],1446,1446
9,wcci_aib,wcci_aib_01_flat_local_t010_topo003_72x576_s0,finished,1,[],1446,1446


## Labels and Run Helpers

In [3]:
def safe_name(text):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-") or "plot"


def save_figure(fig, name):
    if fig is None or not SAVE_FIGURES:
        return None
    path = FIG_DIR / f"{safe_name(name)}.html"
    fig.write_html(path, include_plotlyjs="cdn")
    print("Saved:", path)
    return path


def seed_from_run(run_name):
    match = re.search(r"_s(\d+)$", str(run_name))
    return int(match.group(1)) if match else np.nan


def family_from_run(run_name):
    return re.sub(r"_s\d+$", "", str(run_name))


WCCI_LABELS = {
    "wcci_reduced_mlp_baseline": "reduced MLP baseline 24h",
    "wcci_reduced_mlp_baseline_72x576_s0": "baseline 60M old name",
    "wcci_reduced_mlp_baseline_72x576": "reduced MLP baseline 60M",
    "wcci_reduced_mlp_baseline_72x576_lr20m_f010": "reduced MLP baseline lr20m f0.10",
    "wcci_mlp_baseline_72x576_20M": "MLP baseline 20M",
    "wcci_mlp_baseline_72x576_60M": "MLP baseline 60M",
    "wcci_hvg_01_eval_rho090_72x576": "global rho heuristic 0.90",
    "wcci_hvg_04_eval_local_rho090_72x576": "local rho heuristic 0.90",
    "wcci_hvg_02_gate_final_map_72x576": "gate final-action MAP",
    "wcci_hvg_03_gate_hierarchical_72x576": "gate hierarchical greedy",
    "wcci_sparse16_flat_p003_72x576": "Sparse16 flat p0.003",
    "wcci_aib_00_flat_local_t020_72x576": "AIB flat local t0.20",
    "wcci_aib_00_flat_local_t020_72x576_lr20m_f010": "AIB flat local t0.20 lr20m f0.10",
    "wcci_aib_01_flat_local_t010_72x576": "AIB flat local t0.10",
    "wcci_aib_02_flat_local_t035_72x576": "AIB flat local t0.35",
    "wcci_aib_03_gate_hgreedy_sep_local_t020_72x576": "AIB gate h-greedy t0.20",
    "wcci_aib_04_flat_nonidle_t020_72x576": "AIB flat non-idle t0.20",
    "wcci_aib_01_flat_local_t010_topo003_72x576": "AIB t0.10 topo0.003",
    "wcci_aib_01_flat_local_t010_topo010_72x576": "AIB t0.10 topo0.010",
}

WCCI_COLOR = {
    "MLP baseline 20M": "#1f77b4",
    "MLP baseline 60M": "#4e79a7",
    "reduced MLP baseline 60M": "#1f77b4",
    "reduced MLP baseline lr20m f0.10": "#17becf",
    "global rho heuristic 0.90": "#ff7f0e",
    "local rho heuristic 0.90": "#2ca02c",
    "gate final-action MAP": "#9467bd",
    "gate hierarchical greedy": "#d62728",
    "Sparse16 flat p0.003": "#8c564b",
    "AIB flat local t0.20": "#ff7f0e",
    "AIB flat local t0.20 lr20m f0.10": "#bcbd22",
    "AIB flat local t0.10": "#2ca02c",
    "AIB flat local t0.35": "#d62728",
    "AIB gate h-greedy t0.20": "#8c564b",
    "AIB flat non-idle t0.20": "#9467bd",
    "AIB t0.10 topo0.003": "#e377c2",
    "AIB t0.10 topo0.010": "#7f7f7f",
}


def pretty_label(family):
    return WCCI_LABELS.get(str(family), str(family).replace("wcci_", "").replace("_", " "))


def seeded(prefix, seeds=SEEDS):
    return [f"{prefix}_s{seed}" for seed in seeds]


def available_run_names(history=None):
    data = history_df if history is None else history
    return set(data["run_name"].dropna().astype(str).unique())


def report_missing_runs(run_groups, history=None):
    available = available_run_names(history)
    rows = []
    for label, runs in run_groups.items():
        missing = [run for run in runs if run not in available]
        rows.append({"curve": label, "expected": len(runs), "available": len(runs) - len(missing), "missing": missing})
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


def infer_family_table():
    rows = []
    for run_name in sorted(history_df["run_name"].dropna().astype(str).unique()):
        rows.append({
            "download_group": history_df.loc[history_df["run_name"].eq(run_name), "download_group"].iloc[0],
            "run_name": run_name,
            "family": family_from_run(run_name),
            "label": pretty_label(family_from_run(run_name)),
            "seed": seed_from_run(run_name),
        })
    return pd.DataFrame(rows)


family_table = infer_family_table()
display(family_table.sort_values(["download_group", "family", "seed"]))


,download_group,run_name,family,label,seed
3,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s0,wcci_aib_00_flat_local_t020_72x576,AIB flat local t0.20,0
4,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s1,wcci_aib_00_flat_local_t020_72x576,AIB flat local t0.20,1
5,wcci_aib,wcci_aib_00_flat_local_t020_72x576_s2,wcci_aib_00_flat_local_t020_72x576,AIB flat local t0.20,2
0,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s0,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,0
1,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s1,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,1
2,wcci_aib,wcci_aib_00_flat_local_t020_72x576_lr20m_f010_s2,wcci_aib_00_flat_local_t020_72x576_lr20m_f010,AIB flat local t0.20 lr20m f0.10,2
6,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s0,wcci_aib_01_flat_local_t010_72x576,AIB flat local t0.10,0
7,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s1,wcci_aib_01_flat_local_t010_72x576,AIB flat local t0.10,1
8,wcci_aib,wcci_aib_01_flat_local_t010_72x576_s2,wcci_aib_01_flat_local_t010_72x576,AIB flat local t0.10,2
9,wcci_aib,wcci_aib_01_flat_local_t010_topo003_72x576_s0,wcci_aib_01_flat_local_t010_topo003_72x576,AIB t0.10 topo0.003,0


## Action Distribution Helpers

In [4]:
ACTION0_METRICS = {
    "test": ["test/explain/frac_action_0_agent_{agent}"],
    "train_eval": ["train_eval/explain/frac_action_0_agent_{agent}"],
    "train": ["train/frac_action_0_agent_{agent}"],
}
SURVIVAL_METRICS_FOR_TRADEOFF = {
    "test": ["test/episodic_survival", "test/charts/episodic_survival"],
    "train_eval": ["train_eval/episodic_survival", "train_eval/charts/episodic_survival"],
}
NONIDLE_COUNT_METRICS = [f"train/non_idle_agents_count_{idx}_frac" for idx in range(5)]


def _agent_metric_id(agent):
    text = str(agent)
    return text.rsplit("_", 1)[-1] if text.startswith("agent_") else text


def existing_metric(candidates, *, agent=None, data=None):
    data = history_df if data is None else data
    for template in candidates:
        metric = template.format(agent=_agent_metric_id(agent)) if agent is not None else template
        if metric in data.columns:
            return metric
    return None


def value_scale(values, *, percent=True):
    numeric = pd.to_numeric(values, errors="coerce")
    max_value = numeric.max(skipna=True)
    if pd.isna(max_value):
        return 100.0 if percent else 1.0
    return 100.0 if percent and max_value <= 1.5 else 1.0


def latest_non_null(run_data, metric):
    if metric not in run_data.columns:
        return None
    values = run_data[["_step", metric]].dropna(subset=[metric]).sort_values("_step")
    if values.empty:
        return None
    row = values.iloc[-1]
    return float(row[metric]), int(row["_step"])


def collect_latest_action0(*, source="test", history=None):
    data = history_df if history is None else history
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        family = family_from_run(run_name)
        seed = seed_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        for agent in AGENTS:
            metric = existing_metric(ACTION0_METRICS[source], agent=agent, data=run_data)
            if metric is None:
                continue
            latest = latest_non_null(run_data, metric)
            if latest is None:
                continue
            value, step = latest
            rows.append({
                "run_name": run_name,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "download_group": download_group,
                "agent": agent,
                "metric_source": source,
                "metric": metric,
                "action0_fraction": value,
                "selected_metric_step": step,
            })
    return pd.DataFrame(rows)


def summarize_action0(action_rows):
    if action_rows.empty:
        return pd.DataFrame()
    summary = (
        action_rows.groupby(["condition_label", "family", "agent"], as_index=False, observed=True)
        .agg(
            mean_action0_fraction=("action0_fraction", "mean"),
            std_action0_fraction=("action0_fraction", "std"),
            min_action0_fraction=("action0_fraction", "min"),
            max_action0_fraction=("action0_fraction", "max"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
            selected_metric_steps=("selected_metric_step", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    summary["std_action0_fraction"] = summary["std_action0_fraction"].fillna(0.0)
    return summary


def collect_latest_survival(*, split="test", history=None):
    data = history_df if history is None else history
    candidates = SURVIVAL_METRICS_FOR_TRADEOFF[split]
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        metric = existing_metric(candidates, data=run_data)
        if metric is None:
            continue
        latest = latest_non_null(run_data, metric)
        if latest is None:
            continue
        value, step = latest
        scale = value_scale(run_data[metric], percent=True)
        family = family_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        rows.append({
            "run_name": run_name,
            "family": family,
            "download_group": download_group,
            "condition_label": pretty_label(family),
            "seed": seed_from_run(run_name),
            "survival_percent": value * scale,
            "survival_metric": metric,
            "selected_survival_step": step,
        })
    return pd.DataFrame(rows)


def _families_from_run_groups(run_groups):
    families = []
    for runs in run_groups.values():
        for run in runs:
            family = family_from_run(run)
            if family not in families:
                families.append(family)
    return families


def _order_labels(labels):
    return sorted(labels, key=lambda label: (list(WCCI_COLOR).index(label) if label in WCCI_COLOR else 999, label))


def plot_action0_by_agent(action_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    rows = action_rows[action_rows["family"].isin(families)].copy()
    if rows.empty:
        print(f"No action-0 rows for {title}")
        return None, rows, pd.DataFrame()
    summary = summarize_action0(rows)
    label_order = [label for label in run_groups.keys() if label in set(summary["condition_label"])]
    label_order += [label for label in _order_labels(summary["condition_label"].unique()) if label not in label_order]
    agent_order = [agent for agent in AGENTS if agent in set(summary["agent"])]

    fig = px.bar(
        summary,
        x="condition_label",
        y="mean_action0_fraction",
        color="agent",
        barmode="group",
        error_y="std_action0_fraction",
        category_orders={"condition_label": label_order, "agent": agent_order},
        text=summary["mean_action0_fraction"].map(lambda value: f"{value:.3f}"),
        hover_data={
            "family": True,
            "agent": True,
            "mean_action0_fraction": ":.4f",
            "std_action0_fraction": ":.4f",
            "n_seeds": True,
            "seeds": True,
            "selected_metric_steps": True,
        },
        labels={"condition_label": "run", "mean_action0_fraction": "action 0 fraction", "agent": "agent"},
        title=title,
    )
    for seed in sorted(rows["seed"].dropna().astype(int).unique()):
        seed_rows = rows[rows["seed"].eq(seed)]
        fig.add_trace(go.Scatter(
            x=seed_rows["condition_label"],
            y=seed_rows["action0_fraction"],
            mode="markers",
            name=f"seed {seed}",
            marker={"size": 6, "symbol": "circle-open", "color": "rgba(20,20,20,0.55)"},
            showlegend=False,
            customdata=np.stack([seed_rows["run_name"], seed_rows["agent"], seed_rows["selected_metric_step"]], axis=-1),
            hovertemplate="run=%{customdata[0]}<br>agent=%{customdata[1]}<br>step=%{customdata[2]}<br>action0=%{y:.4f}<extra></extra>",
        ))
    fig.update_layout(
        template="plotly_white",
        width=1450,
        height=650,
        yaxis_range=[0, 1.02],
        legend={"orientation": "v", "yanchor": "top", "y": 1, "xanchor": "left", "x": 1.01},
        margin={"l": 80, "r": 250, "t": 90, "b": 120},
    )
    fig.update_xaxes(tickangle=-25)
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, rows, summary


def collect_nonidle_count_distribution(history=None):
    data = history_df if history is None else history
    rows = []
    for run_name, run_data in data.groupby("run_name", sort=False):
        family = family_from_run(run_name)
        seed = seed_from_run(run_name)
        download_group = run_data["download_group"].iloc[0] if "download_group" in run_data.columns else None
        for metric in NONIDLE_COUNT_METRICS:
            if metric not in run_data.columns:
                continue
            latest = latest_non_null(run_data, metric)
            if latest is None:
                continue
            value, step = latest
            count = int(re.search(r"count_(\d+)_frac", metric).group(1))
            rows.append({
                "run_name": run_name,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "download_group": download_group,
                "nonidle_agent_count": count,
                "fraction": value,
                "selected_metric_step": step,
            })
    return pd.DataFrame(rows)


def plot_nonidle_distribution(nonidle_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    rows = nonidle_rows[nonidle_rows["family"].isin(families)].copy()
    if rows.empty:
        print(f"No non-idle count rows for {title}")
        return None, rows, pd.DataFrame()
    summary = (
        rows.groupby(["condition_label", "family", "nonidle_agent_count"], as_index=False, observed=True)
        .agg(
            mean_fraction=("fraction", "mean"),
            std_fraction=("fraction", "std"),
            n_seeds=("seed", "nunique"),
        )
    )
    summary["std_fraction"] = summary["std_fraction"].fillna(0.0)
    fig = px.bar(
        summary,
        x="condition_label",
        y="mean_fraction",
        color="nonidle_agent_count",
        barmode="group",
        error_y="std_fraction",
        labels={"condition_label": "run", "mean_fraction": "fraction", "nonidle_agent_count": "# non-idle agents"},
        title=title,
    )
    fig.update_layout(template="plotly_white", width=1450, height=620, margin={"l": 80, "r": 220, "t": 90, "b": 120})
    fig.update_xaxes(tickangle=-25)
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, rows, summary


def plot_action0_survival_tradeoff(action_rows, survival_rows, run_groups, *, title, save_name=None):
    families = set(_families_from_run_groups(run_groups))
    action = action_rows[action_rows["family"].isin(families)].copy()
    survival = survival_rows[survival_rows["family"].isin(families)].copy()
    if action.empty or survival.empty:
        print(f"Missing action or survival rows for {title}")
        return None, pd.DataFrame()
    action_all = (
        action.groupby(["run_name", "family", "condition_label", "seed"], as_index=False, observed=True)
        .agg(action0_all_agents=("action0_fraction", "mean"))
    )
    merged = action_all.merge(survival, on=["run_name", "family", "condition_label", "seed"], how="inner")
    if merged.empty:
        print(f"No merged action/survival rows for {title}")
        return None, merged
    mean_rows = (
        merged.groupby(["family", "condition_label"], as_index=False, observed=True)
        .agg(
            mean_action0_all_agents=("action0_all_agents", "mean"),
            std_action0_all_agents=("action0_all_agents", "std"),
            mean_survival_percent=("survival_percent", "mean"),
            std_survival_percent=("survival_percent", "std"),
            n_seeds=("seed", "nunique"),
            seeds=("seed", lambda values: sorted(pd.Series(values).dropna().astype(int).unique().tolist())),
        )
    )
    fig = px.scatter(
        mean_rows,
        x="mean_action0_all_agents",
        y="mean_survival_percent",
        color="condition_label",
        text="condition_label",
        error_x="std_action0_all_agents",
        error_y="std_survival_percent",
        hover_data={"family": True, "n_seeds": True, "seeds": True},
        labels={"mean_action0_all_agents": "avg action-0 fraction", "mean_survival_percent": "test episodic survival (%)", "condition_label": "run"},
        title=title,
    )
    fig.update_traces(marker={"size": 9}, textposition="top center")
    fig.update_layout(template="plotly_white", width=1050, height=680, margin={"l": 80, "r": 220, "t": 90, "b": 70})
    save_figure(fig, save_name or title)
    if SHOW_FIGURES:
        fig.show()
    return fig, mean_rows.sort_values("mean_survival_percent", ascending=False)


def load_full_test_action_summaries():
    root = TASK_DIR / "outputs" / "full_test_eval_actions"
    rows = []
    if not root.exists():
        return pd.DataFrame()
    for path in sorted(root.glob("*/action_summary.json")):
        run_like = path.parent.name
        if "wcci" not in run_like.lower():
            continue
        try:
            payload = json.loads(path.read_text())
        except Exception as exc:
            print(f"Skipped {path}: {exc}")
            continue
        stem = re.sub(r"_step\d+_job\d+$", "", run_like)
        stem = stem.removeprefix("best_test_")
        family = family_from_run(stem)
        seed = seed_from_run(stem)
        for agent, agent_data in payload.get("agents", {}).items():
            rows.append({
                "run_like": run_like,
                "run_name": stem,
                "family": family,
                "condition_label": pretty_label(family),
                "seed": seed,
                "agent": agent,
                "action0_fraction": agent_data.get("action0_fraction"),
                "global_step": payload.get("global_step"),
                "path": str(path),
            })
    return pd.DataFrame(rows)


## Editable WCCI Run Groups

In [5]:
# Edit this cell to choose which WCCI runs appear in each action plot.
BASELINE_FOR_COMPARISONS = "wcci_reduced_mlp_baseline_72x576_lr20m_f010"
BASELINE_LABEL = "reduced MLP baseline lr20m f0.10"
ACTION_METRIC_SOURCE = "test"  # "test", "train_eval", or "train"

WCCI_BASELINE_RUNS = {
    "MLP baseline 20M": seeded("wcci_mlp_baseline_72x576_20M"),
    "MLP baseline 60M": seeded("wcci_mlp_baseline_72x576_60M"),
    "reduced MLP baseline lr20m f0.10": seeded("wcci_reduced_mlp_baseline_72x576_lr20m_f010"),
}
WCCI_HVG_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "global rho heuristic 0.90": seeded("wcci_hvg_01_eval_rho090_72x576"),
    "local rho heuristic 0.90": seeded("wcci_hvg_04_eval_local_rho090_72x576"),
    "gate final-action MAP": seeded("wcci_hvg_02_gate_final_map_72x576"),
    "gate hierarchical greedy": seeded("wcci_hvg_03_gate_hierarchical_72x576"),
}
WCCI_SPARSE16_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "Sparse16 flat p0.003": seeded("wcci_sparse16_flat_p003_72x576"),
}
WCCI_AIB_24H_RUNS = {
    "reduced MLP baseline 24h": seeded("wcci_reduced_mlp_baseline"),
    "AIB flat local t0.20": seeded("wcci_aib_00_flat_local_t020_72x576"),
    "AIB flat local t0.10": seeded("wcci_aib_01_flat_local_t010_72x576"),
    "AIB flat local t0.35": seeded("wcci_aib_02_flat_local_t035_72x576"),
}
WCCI_AIB_RUNS = {
    BASELINE_LABEL: seeded(BASELINE_FOR_COMPARISONS),
    "AIB flat local t0.20": seeded("wcci_aib_00_flat_local_t020_72x576"),
    "AIB flat local t0.20 lr20m f0.10": seeded("wcci_aib_00_flat_local_t020_72x576_lr20m_f010"),
    "AIB flat local t0.10": seeded("wcci_aib_01_flat_local_t010_72x576"),
    "AIB t0.10 topo0.003": seeded("wcci_aib_01_flat_local_t010_topo003_72x576"),
    "AIB t0.10 topo0.010": seeded("wcci_aib_01_flat_local_t010_topo010_72x576"),
}
ALL_COMPARISONS = {
    "WCCI baselines": WCCI_BASELINE_RUNS,
    "WCCI HVG": WCCI_HVG_RUNS,
    "WCCI Sparse16": WCCI_SPARSE16_RUNS,
    "WCCI AIB 24h": WCCI_AIB_24H_RUNS,
    "WCCI AIB": WCCI_AIB_RUNS,
}

# Some AIB run names exist in both wcci_aib_24h and wcci_aib.
# Always filter by the downloaded source folder as well as run name.
COMPARISON_SOURCE_GROUPS = {
    "WCCI baselines": ["wcci_baseline"],
    "WCCI HVG": ["wcci_baseline", "wcci_hvg"],
    "WCCI Sparse16": ["wcci_baseline", "wcci_sparse16"],
    "WCCI AIB 24h": ["wcci_aib_24h"],
    "WCCI AIB": ["wcci_baseline", "wcci_aib"],
}


def source_history(comparison_name):
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return history_df[history_df["download_group"].isin(sources)].copy()


def source_rows(rows, comparison_name):
    if rows is None or rows.empty or "download_group" not in rows.columns:
        return rows
    sources = COMPARISON_SOURCE_GROUPS.get(comparison_name, WCCI_GROUPS)
    return rows[rows["download_group"].isin(sources)].copy()


for name, groups in ALL_COMPARISONS.items():
    print(f"\n{name}")
    report_missing_runs(groups, source_history(name))



WCCI baselines


,curve,expected,available,missing
0,MLP baseline 20M,3,3,[]
1,MLP baseline 60M,3,3,[]
2,reduced MLP baseline lr20m f0.10,3,3,[]



WCCI HVG


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,3,[]
1,global rho heuristic 0.90,3,3,[]
2,local rho heuristic 0.90,3,3,[]
3,gate final-action MAP,3,3,[]
4,gate hierarchical greedy,3,3,[]



WCCI Sparse16


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,3,[]
1,Sparse16 flat p0.003,3,3,[]



WCCI AIB 24h


,curve,expected,available,missing
0,reduced MLP baseline 24h,3,3,[]
1,AIB flat local t0.20,3,3,[]
2,AIB flat local t0.10,3,3,[]
3,AIB flat local t0.35,3,3,[]



WCCI AIB


,curve,expected,available,missing
0,reduced MLP baseline lr20m f0.10,3,3,[]
1,AIB flat local t0.20,3,3,[]
2,AIB flat local t0.20 lr20m f0.10,3,3,[]
3,AIB flat local t0.10,3,3,[]
4,AIB t0.10 topo0.003,3,3,[]
5,AIB t0.10 topo0.010,3,3,[]


## Collect Latest Metrics

In [6]:
action0_latest = collect_latest_action0(source=ACTION_METRIC_SOURCE)
survival_latest = collect_latest_survival(split="test")
nonidle_latest = collect_nonidle_count_distribution()
full_test_action0 = load_full_test_action_summaries()

print(f"Latest action-0 rows from {ACTION_METRIC_SOURCE}: {len(action0_latest)}")
print(f"Latest survival rows: {len(survival_latest)}")
print(f"Latest non-idle-count rows: {len(nonidle_latest)}")
print(f"WCCI full-test action-summary rows: {len(full_test_action0)}")

if full_test_action0.empty:
    print("No WCCI full-test action summaries were found locally. The plots below use W&B history action metrics.")
else:
    display(full_test_action0.sort_values(["condition_label", "seed", "agent"]))


Latest action-0 rows from test: 180
Latest survival rows: 45
Latest non-idle-count rows: 225
WCCI full-test action-summary rows: 0
No WCCI full-test action summaries were found locally. The plots below use W&B history action metrics.


## WCCI Baselines

In [7]:
fig_action0_baselines, action0_baselines_rows, action0_baselines_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI baselines"), WCCI_BASELINE_RUNS,
    title=f"WCCI baselines: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_baselines_action0_{ACTION_METRIC_SOURCE}",
)
fig_action0_baselines

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_baselines_action0_test.html


## WCCI Heuristic vs Gate

In [8]:
fig_action0_hvg, action0_hvg_rows, action0_hvg_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI HVG"), WCCI_HVG_RUNS,
    title=f"WCCI heuristic vs gate: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_hvg_action0_{ACTION_METRIC_SOURCE}",
)
fig_action0_hvg

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_hvg_action0_test.html


## WCCI Sparse16

In [9]:
fig_action0_sparse16, action0_sparse16_rows, action0_sparse16_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI Sparse16"), WCCI_SPARSE16_RUNS,
    title=f"WCCI Sparse16: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_sparse16_action0_{ACTION_METRIC_SOURCE}",
)
fig_action0_sparse16

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_sparse16_action0_test.html


## WCCI AIB 24h

In [10]:
fig_action0_aib24h, action0_aib24h_rows, action0_aib24h_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI AIB 24h"), WCCI_AIB_24H_RUNS,
    title=f"WCCI AIB 24h: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_aib_24h_action0_{ACTION_METRIC_SOURCE}",
)
fig_action0_aib24h

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_aib_24h_action0_test.html


## WCCI AIB

In [11]:
fig_action0_aib, action0_aib_rows, action0_aib_summary = plot_action0_by_agent(
    source_rows(action0_latest, "WCCI AIB"), WCCI_AIB_RUNS,
    title=f"WCCI AIB: action-0 fraction by agent ({ACTION_METRIC_SOURCE})",
    save_name=f"wcci_aib_action0_{ACTION_METRIC_SOURCE}",
)
fig_action0_aib

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_aib_action0_test.html


## Non-Idle Agent Count Distribution

In [12]:
fig_nonidle_hvg, nonidle_hvg_rows, nonidle_hvg_summary = plot_nonidle_distribution(
    source_rows(nonidle_latest, "WCCI HVG"), WCCI_HVG_RUNS,
    title="WCCI HVG: train non-idle-agent count distribution",
    save_name="wcci_hvg_nonidle_count_distribution",
)
fig_nonidle_hvg

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/wcci_hvg_nonidle_count_distribution.html


## Action-0 / Survival Tradeoffs

In [13]:
tradeoff_outputs = {}
for comparison_name, groups in ALL_COMPARISONS.items():
    fig, rows = plot_action0_survival_tradeoff(
        source_rows(action0_latest, comparison_name), source_rows(survival_latest, comparison_name), groups,
        title=f"{comparison_name}: action-0 / survival tradeoff",
        save_name=f"{comparison_name}_action0_survival_tradeoff",
    )
    tradeoff_outputs[comparison_name] = rows.assign(comparison=comparison_name) if not rows.empty else rows

combined_tradeoff = pd.concat([df for df in tradeoff_outputs.values() if isinstance(df, pd.DataFrame) and not df.empty], ignore_index=True) if tradeoff_outputs else pd.DataFrame()
display(combined_tradeoff)

Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_baselines_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_HVG_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_Sparse16_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_AIB_24h_action0_survival_tradeoff.html


Saved: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wcci_metric_figures/WCCI_AIB_action0_survival_tradeoff.html


,family,condition_label,mean_action0_all_agents,std_action0_all_agents,mean_survival_percent,std_survival_percent,n_seeds,seeds,comparison
0,wcci_mlp_baseline_72x576_60M,MLP baseline 60M,0.163508,0.038339,40.143058,11.626336,3,"[0, 1, 2]",WCCI baselines
1,wcci_reduced_mlp_baseline_72x576_lr20m_f010,reduced MLP baseline lr20m f0.10,0.189355,0.061226,39.678740,6.270455,3,"[0, 1, 2]",WCCI baselines
2,wcci_mlp_baseline_72x576_20M,MLP baseline 20M,0.191391,0.030786,35.356405,2.458849,3,"[0, 1, 2]",WCCI baselines
3,wcci_hvg_01_eval_rho090_72x576,global rho heuristic 0.90,0.986005,0.005737,49.517489,9.282226,3,"[0, 1, 2]",WCCI HVG
4,wcci_reduced_mlp_baseline_72x576_lr20m_f010,reduced MLP baseline lr20m f0.10,0.189355,0.061226,39.678740,6.270455,3,"[0, 1, 2]",WCCI HVG
5,wcci_hvg_03_gate_hierarchical_72x576,gate hierarchical greedy,0.056590,0.009124,36.217647,7.444938,3,"[0, 1, 2]",WCCI HVG
6,wcci_hvg_02_gate_final_map_72x576,gate final-action MAP,0.482986,0.029093,32.291822,8.314238,3,"[0, 1, 2]",WCCI HVG
7,wcci_hvg_04_eval_local_rho090_72x576,local rho heuristic 0.90,0.998114,0.000217,14.518730,3.558794,3,"[0, 1, 2]",WCCI HVG
8,wcci_reduced_mlp_baseline_72x576_lr20m_f010,reduced MLP baseline lr20m f0.10,0.189355,0.061226,39.678740,6.270455,3,"[0, 1, 2]",WCCI Sparse16
9,wcci_sparse16_flat_p003_72x576,Sparse16 flat p0.003,0.234700,0.008890,37.610188,7.694445,3,"[0, 1, 2]",WCCI Sparse16


## Optional Full-Test Action Summaries

In [14]:
if full_test_action0.empty:
    print("No WCCI full-test action summaries found. Re-run this cell after adding outputs/full_test_eval_actions/*wcci*/action_summary.json.")
else:
    full_test_summary = summarize_action0(full_test_action0.rename(columns={"run_like": "run_name"}))
    display(full_test_summary.sort_values(["condition_label", "agent"]))
    fig_full_test = px.bar(
        full_test_summary,
        x="condition_label",
        y="mean_action0_fraction",
        color="agent",
        barmode="group",
        error_y="std_action0_fraction",
        title="WCCI full-test action-0 fraction by agent",
        labels={"condition_label": "run", "mean_action0_fraction": "action 0 fraction", "agent": "agent"},
    )
    fig_full_test.update_layout(template="plotly_white", width=1450, height=650, yaxis_range=[0, 1.02])
    fig_full_test.update_xaxes(tickangle=-25)
    save_figure(fig_full_test, "wcci_full_test_action0_by_agent")
    fig_full_test

No WCCI full-test action summaries found. Re-run this cell after adding outputs/full_test_eval_actions/*wcci*/action_summary.json.
